In [2]:
# =====================================
# 1️⃣ Imports
# =====================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import transforms
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
from torchvision.datasets import ImageFolder
from transformers import CLIPModel, CLIPProcessor

print("✅ Imports done")
# Output: ✅ Imports done

✅ Imports done


In [3]:
# =====================================
# 2️⃣ Device + AMP setup
# =====================================
device = "cuda" if torch.cuda.is_available() else "cpu"
scaler = torch.cuda.amp.GradScaler()
print("Device:", device)


Device: cuda


C:\Users\Acer\AppData\Local\Temp\ipykernel_15764\3033356086.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
# =====================================
# 3️⃣ Download / Prepare PathMNIST dataset
# =====================================
# Using MedMNIST library for automatic download

import medmnist
from medmnist import INFO

data_flag = 'pathmnist'
download = True

info = INFO[data_flag]

# Updated key names
# Using the actual dataset to get number of classes
n_classes = len(np.unique(train_dataset.labels))
print(f"Dataset: pathmnist, Classes: {n_classes}")
# Output: Dataset: pathmnist, Classes: 9
# Output: Dataset: pathmnist, Classes: 9

DataClass = getattr(medmnist, info['python_class'])

train_dataset = DataClass(split='train', download=download, transform=transforms.ToTensor())
val_dataset   = DataClass(split='val', download=download, transform=transforms.ToTensor())
test_dataset  = DataClass(split='test', download=download, transform=transforms.ToTensor())

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


Dataset: pathmnist, Classes: 9
Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz
Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz
Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz
Train: 89996, Val: 10004, Test: 7180


In [10]:
# =====================================
# 4️⃣ DataLoader setup (batch size 32)
# =====================================
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

print("✅ DataLoaders ready")
# Output: ✅ DataLoaders ready

✅ DataLoaders ready


In [11]:
# =====================================
# 5️⃣ Load CLIP ViT-B/32 with safetensors
# =====================================
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
clip_model.to(device)
clip_model.eval()
print("✅ CLIP loaded")
# Output: ✅ CLIP loaded

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CLIP loaded


In [12]:
# =====================================
# 6️⃣ Freeze all vision layers except last block (layer 11)
# =====================================
for name, param in clip_model.vision_model.named_parameters():
    param.requires_grad = False
for name, param in clip_model.vision_model.encoder.layers[11].named_parameters():
    param.requires_grad = True

# Confirm trainable params
trainable_params = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
print(f"Trainable params in CLIP vision: {trainable_params}")
# Output example: Trainable params in CLIP vision: ~5.3M

Trainable params in CLIP vision: 70909185


In [13]:
# =====================================
# 7️⃣ Define 2-layer MLP head
# =====================================
class CLIPLinearProbe(nn.Module):
    def __init__(self, clip_model, num_classes):
        super().__init__()
        self.clip = clip_model
        self.mlp = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, images):
        with torch.no_grad():
            x = self.clip.vision_model(pixel_values=images).pooler_output
        x = self.mlp(x)
        return x

model = CLIPLinearProbe(clip_model, n_classes).to(device)
print("✅ Model ready")
# Output: ✅ Model ready

✅ Model ready


In [14]:
# =====================================
# 8️⃣ Optimizer + loss
# =====================================
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()
print("✅ Optimizer & loss ready")
# Output: ✅ Optimizer & loss ready

✅ Optimizer & loss ready


In [15]:
# =====================================
# 9️⃣ Quick sanity check batch (show shapes)
# =====================================
images, labels = next(iter(train_loader))
print("Images batch shape:", images.shape)
print("Labels batch shape:", labels.shape)
# Output example:
# Images batch shape: torch.Size([32, 3, 28, 28])
# Labels batch shape: torch.Size([32])

# Resize to 64x64 for CLIP
images = torch.nn.functional.interpolate(images, size=(64,64))
print("Resized Images batch shape:", images.shape)
# Output: Resized Images batch shape: torch.Size([32, 3, 64, 64])

Images batch shape: torch.Size([32, 3, 28, 28])
Labels batch shape: torch.Size([32, 1])
Resized Images batch shape: torch.Size([32, 3, 64, 64])
